In [ ]:
# create and save PSTH
from nwb_utils import NWBUtils
from behavior_utils import extract_fitted_data, find_trials,get_fitted_model_names,generate_behavior_summary
from create_psth import extract_neuron_psth_to_zarr

session_name='764787_2024-12-11_15-01-15_sorted_2025-02-21_17-11-57'
#session_name='764790_2024-12-19_16-11-34'
print(get_fitted_model_names(session_name=session_name))

nwb_data,tag=NWBUtils.combine_nwb(session_name=session_name)

psth_da = extract_neuron_psth_to_zarr(
    nwb_data       = nwb_data,
    align_to_event = ["go_cue","reward_go_cue_start"],
    time_window    = (-6, 6),
    bin_size       = 0.1,
    save_folder    = "/root/capsule/results",
    save_name      = "psth_mouse123",
)

In [ ]:
from create_psth import  load_psth_raster_from_zarr, mean_firing_rate_matrix,load_zarr
from behavior_utils import find_trials

# 0.  Load the pre‑computed PSTH cube
#psth_da = load_psth_raster_from_zarr("/root/capsule/results/psth_mouse123.zarr",align_to_event='go_cue',load_type='both')
psth_full = load_zarr("/root/capsule/results/psth_mouse123.zarr")

In [ ]:
from create_psth import load_psth_raster_subset
psth,raster=load_psth_raster_subset(psth_full,align_to_event='go_cue')

In [3]:
from joblib import Parallel, delayed
import os, traceback, pandas as pd
from nwb_utils import NWBUtils
from create_psth import extract_neuron_psth_to_zarr
from general_utils import find_ephys_sessions

SAVE_DIR = "/root/capsule/scratch/psth_results/"
ALIGN_TO = ["go_cue","reward_go_cue_start"]
TIME_WINDOW = (-6, 6)
BIN_SIZE = 0.2
RETRIES = 0  # set >0 if you want automatic retries on failure

def worker(session_name):
    """Return a status dict; never raise so Parallel keeps running."""
    try:
        nwb_data, _ = NWBUtils.combine_nwb(session_name=session_name)
        save_name = session_name+'_'+str(BIN_SIZE)+'s'

        attempt = 0
        while True:
            try:
                extract_neuron_psth_to_zarr(
                    nwb_data=nwb_data,
                    align_to_event=ALIGN_TO,
                    time_window=TIME_WINDOW,
                    bin_size=BIN_SIZE,
                    save_folder=SAVE_DIR,
                    save_name=save_name,
                )
                return {"session": session_name, "ok": True, "error": "", "message": ""}
            except Exception as e:
                if attempt < RETRIES:
                    attempt += 1
                    continue
                # final failure
                return {
                    "session": session_name,
                    "ok": False,
                    "error": type(e).__name__,
                    "message": str(e),
                    # keep traceback short; remove if too noisy
                    "traceback": "".join(traceback.format_exception_only(type(e), e)).strip(),
                }
    except Exception as e:
        return {
            "session": session_name,
            "ok": False,
            "error": type(e).__name__,
            "message": str(e),
            "traceback": "".join(traceback.format_exception_only(type(e), e)).strip(),
        }

# discover sessions
sessions = find_ephys_sessions()
session_list = list(sessions[2])

#session_list =['ecephys_764787_2024-12-10_13-42-03_sorted_2025-01-24_15-52-45']
# choose parallelism (respect Code Ocean CPU limit if present)
n_jobs = max(1, (int(os.getenv("CO_CPUS", os.cpu_count() or 2)) - 1))

# run
results = Parallel(n_jobs=n_jobs, backend="loky")(delayed(worker)(s) for s in session_list)

# summarize
ok = [r for r in results if r["ok"]]
fail = [r for r in results if not r["ok"]]

print(f"Succeeded: {len(ok)} / {len(results)}")
if fail:
    print(f"Failed: {len(fail)} (details saved to {SAVE_DIR}/failed_sessions.csv)")
    # save failures so they’re captured as a data asset
    pd.DataFrame(fail)[["session", "error", "message"]].to_csv(
        os.path.join(SAVE_DIR, "failed_sessions.csv"), index=False
    )


Found behavior NWB: /root/capsule/data/behavior_nwb/764787_2024-12-10_13-42-03.nwb


/opt/conda/lib/python3.10/site-packages/hdmf/spec/namespace.py:590: UserWarning: Ignoring the following cached namespace(s) because another version is already loaded:
core - cached version: 2.6.0-alpha, loaded version: 2.7.0
The loaded extension(s) may not be compatible with the cached extension(s) in the file. Please check the extension documentation and ignore this warning if these versions are compatible.
  self.warn_for_ignored_namespaces(ignored_namespaces)


Successfully read behavior NWB from: /root/capsule/data/behavior_nwb/764787_2024-12-10_13-42-03.nwb
Succeeded: 0 / 1
Failed: 1 (details saved to /root/capsule/scratch/psth_results//failed_sessions.csv)


In [2]:
session_list

['ecephys_753124_2024-12-10_17-24-56_sorted_2024-12-13_09-48-25',
 'ecephys_753125_2024-10-09_10-50-19_sorted_2024-11-09_20-03-58',
 'ecephys_753125_2024-10-10_14-41-23_sorted_2024-11-09_20-18-36',
 'ecephys_753125_2024-10-14_15-37-15_sorted_2024-11-09_20-07-38',
 'ecephys_753125_2024-10-15_16-16-22_sorted_2024-11-09_19-57-50',
 'ecephys_753126_2024-10-10_17-51-24_sorted_2025-02-21_13-56-40',
 'ecephys_753126_2024-10-11_13-14-24_sorted_2024-11-09_19-18-21',
 'ecephys_753126_2024-10-15_12-20-35_sorted_2024-11-09_19-47-57',
 'ecephys_764769_2024-12-11_18-21-49_sorted_2024-12-13_10-04-48',
 'ecephys_764769_2024-12-12_16-05-00_sorted_2024-12-13_10-34-23',
 'ecephys_764769_2024-12-13_15-41-07_sorted_2024-12-17_18-00-23',
 'ecephys_764787_2024-12-10_13-42-03_sorted_2025-01-24_15-52-45',
 'ecephys_764787_2024-12-11_15-01-15_sorted_2025-02-21_17-11-57',
 'ecephys_764787_2024-12-12_11-54-14_sorted_2024-12-13_10-39-18',
 'ecephys_764787_2024-12-13_18-27-42_sorted_2024-12-17_18-31-24',
 'ecephys_